# TD 1 — Fondements mathématiques : NumPy, softmax, descente de gradient

**Objectifs :**
1. (Re)prendre en main NumPy : vecteurs, matrices, broadcasting.
2. Implémenter softmax, entropie et entropie croisée *à la main*.
3. Coder une descente de gradient et analyser sa convergence.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)  # graine fixée : reproductibilité !

## 1. Vecteurs et matrices avec NumPy

Un vecteur est un `np.array` à 1 dimension, une matrice à 2 dimensions.

In [ ]:
u = np.array([1.0, 2.0, 3.0])
v = np.array([-1.0, 0.5, 2.0])

print("u + v      =", u + v)          # addition terme à terme
print("2 * u      =", 2 * u)          # multiplication par un scalaire
print("u * v      =", u * v)          # ATTENTION : produit terme à terme, PAS scalaire
print("u @ v      =", u @ v)          # produit scalaire <u, v>
print("norme de u =", np.linalg.norm(u))

**Exercice 1.1** — Écrivez une fonction `cosinus(u, v)` qui calcule la similarité
cosinus (cf. CM). Vérifiez que `cosinus(u, u) == 1` et que deux vecteurs
orthogonaux donnent 0.

In [ ]:
def cosinus(u, v):
    # À COMPLÉTER
    return u @ v / (np.linalg.norm(u) * np.linalg.norm(v))

assert np.isclose(cosinus(u, u), 1.0)
assert np.isclose(cosinus(np.array([1.0, 0.0]), np.array([0.0, 1.0])), 0.0)
print("OK !")

## 2. Matrices et produit matrice-vecteur

In [ ]:
W = rng.normal(size=(4, 3))   # matrice 4x3 : transforme R^3 en R^4
x = rng.normal(size=3)

y = W @ x
print("W a pour forme", W.shape, "; x a pour forme", x.shape, "; W @ x a pour forme", y.shape)

**Exercice 2.1** — Sans exécuter, prédisez la forme de `(A @ B) @ x` avec
`A` de forme (5, 4), `B` de forme (4, 3), `x` de forme (3,). Puis vérifiez.

**Exercice 2.2** — Un « réseau linéaire » à deux couches est $y = A(Bx)$.
Montrez numériquement qu'il existe une matrice unique $C$ telle que $y = Cx$
(c'est l'argument du CM : sans non-linéarité, la profondeur ne sert à rien).

In [ ]:
A = rng.normal(size=(5, 4))
B = rng.normal(size=(4, 3))

C = A @ B  # À COMPLÉTER : quelle est la matrice équivalente ?
assert np.allclose(A @ (B @ x), C @ x)
print("Deux couches linéaires = une seule couche. CQFD.")

## 3. Softmax

$$\mathrm{softmax}(\mathbf{z})_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

**Exercice 3.1** — Implémentez `softmax(z)`. Vérifiez que la sortie est positive
et somme à 1.

**Piège numérique :** `np.exp(1000)` déborde (`inf`). L'astuce standard :
soustraire `max(z)` avant l'exponentielle (le résultat est mathématiquement
identique — prouvez-le sur papier !).

In [ ]:
def softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - z.max()          # stabilité numérique
    e = np.exp(z)
    return e / e.sum()

p = softmax([2.0, 1.0, 0.1])
print("softmax([2, 1, 0.1]) =", p.round(3), "| somme =", p.sum())

# Le test du piège :
print("softmax([1000, 1001]) =", softmax([1000.0, 1001.0]).round(3))  # sans l'astuce : nan !

**Exercice 3.2** — La *température* (on la retrouvera en S6) : tracez
`softmax(z / T)` pour `z = [2, 1, 0.1]` et `T` dans {0.1, 0.5, 1, 2, 10}.
Que se passe-t-il aux extrêmes ?

In [ ]:
z = np.array([2.0, 1.0, 0.1])
for T in [0.1, 0.5, 1.0, 2.0, 10.0]:
    print(f"T = {T:5.1f} ->", softmax(z / T).round(3))

## 4. Entropie et entropie croisée

In [ ]:
def entropie(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]                     # convention 0 log 0 = 0
    return -(p * np.log(p)).sum()

print("Dé équilibré :", entropie(np.ones(6) / 6), "(= log 6 =", np.log(6).round(3), ")")
print("Certitude    :", entropie([1.0, 0, 0, 0, 0, 0]))

**Exercice 4.1** — Implémentez l'entropie croisée dans le cas qui nous intéresse :
le vrai token est d'indice `k`, le modèle prédit la distribution `p`.
La perte vaut $-\log p_k$.

In [ ]:
def cross_entropy(p, k):
    return -np.log(p[k])

p = softmax([2.0, 1.0, 0.1])
print("Le modèle donne p =", p.round(3))
print("Perte si le vrai token est 0 :", cross_entropy(p, 0).round(3))
print("Perte si le vrai token est 2 :", cross_entropy(p, 2).round(3))
# Moralité : plus le modèle se trompe, plus la perte est grande.

## 5. Descente de gradient

On minimise d'abord une fonction 1D : $f(x) = (x - 3)^2 + 1$.
Sa dérivée : $f'(x) = 2(x - 3)$. Le minimum est en $x = 3$.

In [ ]:
f = lambda x: (x - 3) ** 2 + 1
df = lambda x: 2 * (x - 3)

def descente(x0, eta, n_steps):
    xs = [x0]
    for _ in range(n_steps):
        xs.append(xs[-1] - eta * df(xs[-1]))
    return np.array(xs)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
grille = np.linspace(-2, 8, 200)
for ax, eta in zip(axes, [0.05, 0.5, 1.05]):
    xs = descente(x0=-1.5, eta=eta, n_steps=15)
    ax.plot(grille, f(grille), lw=1)
    ax.plot(xs, f(xs), "o-", ms=4)
    ax.set_title(f"eta = {eta}")
plt.suptitle("Trop petit / bien réglé / divergence")
plt.show()

**Exercice 5.1** — Décrivez en une phrase chacun des trois régimes observés.

**Exercice 5.2 (2D)** — Minimisez $f(x, y) = x^2 + 10y^2$ depuis $(8, 3)$.
Calculez le gradient à la main, implémentez, tracez la trajectoire sur les
courbes de niveau (`plt.contour`). Pourquoi la trajectoire zigzague-t-elle ?

In [ ]:
def grad_f2(p):
    x, y = p
    return np.array([2 * x, 20 * y])

p = np.array([8.0, 3.0])
traj = [p.copy()]
for _ in range(40):
    p -= 0.08 * grad_f2(p)
    traj.append(p.copy())
traj = np.array(traj)

X, Y = np.meshgrid(np.linspace(-9, 9, 100), np.linspace(-4, 4, 100))
plt.contour(X, Y, X**2 + 10 * Y**2, levels=20)
plt.plot(traj[:, 0], traj[:, 1], "o-", ms=3, color="crimson")
plt.title("Descente de gradient en 2D")
plt.show()

## 6. Bonus : régression linéaire complète

Tout le deep learning en miniature : données, modèle $\hat y = wx + b$,
perte quadratique, gradients, boucle d'entraînement.

In [ ]:
# Données synthétiques : y = 2.5 x - 1 + bruit
x_data = rng.uniform(-3, 3, size=100)
y_data = 2.5 * x_data - 1 + rng.normal(scale=0.8, size=100)

w, b = 0.0, 0.0
eta = 0.05
pertes = []
for step in range(200):
    y_hat = w * x_data + b
    erreur = y_hat - y_data
    perte = (erreur ** 2).mean()
    pertes.append(perte)
    # gradients (calculez-les sur papier pour vérifier !)
    dw = 2 * (erreur * x_data).mean()
    db = 2 * erreur.mean()
    w -= eta * dw
    b -= eta * db

print(f"Appris : w = {w:.3f}, b = {b:.3f}  (vraies valeurs : 2.5 et -1)")
plt.plot(pertes); plt.xlabel("itération"); plt.ylabel("perte"); plt.title("La perte descend !")
plt.show()

**Pour aller plus loin** — Remplacez la perte quadratique par la perte absolue.
Que devient le gradient ? La convergence ?